# Exercise 1: Next-Word Prediction

In [1]:
import re
from collections import defaultdict, Counter


In [ ]:
#  STEP 1: Load & Preprocess

def preprocess(text):
    """Lowercase, remove punctuation, split into word tokens."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9'\s]", "", text)
    tokens = text.split()
    return tokens

In [3]:
#  STEP 2: Build N-gram Models

def build_unigram(tokens):
    """
    Unigram: count each word independently.
    No context — just word frequencies.
    """
    model = Counter(tokens)
    return model


In [4]:
def build_bigram(tokens):
    """
    Bigram: count pairs (word1 -> word2).
    Context = 1 previous word.
    model["i"] = {"love": 3, "am": 2, ...}
    """
    model = defaultdict(Counter)
    for i in range(len(tokens) - 1):
        word1 = tokens[i]
        word2 = tokens[i + 1]
        model[word1][word2] += 1
    return model

In [5]:
def build_trigram(tokens):
    """
    Trigram: count triplets (word1, word2 -> word3).
    Context = 2 previous words.
    model["i love"] = {"you": 4, "it": 2, ...}
    """
    model = defaultdict(Counter)
    for i in range(len(tokens) - 2):
        context = tokens[i] + " " + tokens[i + 1]   # e.g. "i love"
        next_word = tokens[i + 2]                    # e.g. "you"
        model[context][next_word] += 1
    return model

In [6]:
#STEP 3: Predict Next Word


def predict_unigram(model, top_k=3):
    """
    Unigram prediction — ignores input, returns most frequent words.
    P(word) = count(word) / total_words
    """
    total = sum(model.values())
    top = model.most_common(top_k)
    return [(word, count / total) for word, count in top]


In [7]:
def predict_bigram(model, context_word, top_k=3):
    """
    Bigram prediction — uses last 1 word.
    P(word | context_word) = count(context_word, word) / count(context_word)
    """
    counts = model.get(context_word, None)
    if not counts:
        return []
    total = sum(counts.values())
    top = counts.most_common(top_k)
    return [(word, count / total) for word, count in top]

In [8]:
def predict_trigram(model, word1, word2, top_k=3):
    """
    Trigram prediction — uses last 2 words.
    P(word | word1, word2) = count(word1+word2, word) / count(word1+word2)
    """
    context = word1 + " " + word2
    counts = model.get(context, None)
    if not counts:
        return []
    total = sum(counts.values())
    top = counts.most_common(top_k)
    return [(word, count / total) for word, count in top]

In [9]:
#STEP 4: Print Results Nicely
def show_predictions(label, predictions):
    print(f"\n{'─'*40}")
    print(f"  {label}")
    print(f"{'─'*40}")
    if not predictions:
        print("  (no predictions found)")
        return
    for rank, (word, prob) in enumerate(predictions, 1):
        bar = "█" * int(prob * 30)
        print(f"  {rank}. '{word}' {bar} {prob*100:.1f}%")

In [10]:
#  MAIN

In [10]:
if __name__ == "__main__":

    # Sample corpus (replace with open("input.txt").read() for real data)
    corpus = """
    To be or not to be that is the question whether tis nobler in the mind to suffer
    the slings and arrows of outrageous fortune or to take arms against a sea of troubles
    and by opposing end them to die to sleep no more and by a sleep to say we end
    the heartache and the thousand natural shocks that flesh is heir to tis a consummation
    devoutly to be wished to die to sleep perchance to dream ay there is the rub
    for in that sleep of death what dreams may come when we have shuffled off this mortal coil
    shall i compare thee to a summer day thou art more lovely and more temperate
    rough winds do shake the darling buds of may and summer lease hath all too short a date
    the quality of mercy is not strained it droppeth as the gentle rain from heaven
    all the world is a stage and all the men and women merely players
    they have their exits and their entrances and one man in his time plays many parts
    good night good night parting is such sweet sorrow that i shall say good night
    what light through yonder window breaks it is the east and juliet is the sun
    friends romans countrymen lend me your ears i come to bury caesar not to praise him
    the evil that men do lives after them the good is oft interred with their bones
    tomorrow and tomorrow and tomorrow creeps in this petty pace from day to day
    to the last syllable of recorded time and all our yesterdays have lighted fools
    love is not love which alters when it alteration finds or bends with the remover
    we know what we are but know not what we may be
    brevity is the soul of wit and tediousness the limbs and outward flourishes
    """

    # ── Build ──
    tokens = preprocess(corpus)
    print(f"Tokens: {len(tokens)} | Unique words: {len(set(tokens))}")

    unigram_model = build_unigram(tokens)
    bigram_model  = build_bigram(tokens)
    trigram_model = build_trigram(tokens)

    # ── Unigram: no context needed ──
    show_predictions(
        "UNIGRAM  →  most frequent words overall",
        predict_unigram(unigram_model, top_k=3)
    )

    # ── Bigram: give last 1 word ──
    show_predictions(
        "BIGRAM   |  context: 'the'",
        predict_bigram(bigram_model, context_word="the", top_k=3)
    )

    show_predictions(
        "BIGRAM   |  context: 'to'",
        predict_bigram(bigram_model, context_word="to", top_k=3)
    )

    # ── Trigram: give last 2 words ──
    show_predictions(
        "TRIGRAM  |  context: 'to be'",
        predict_trigram(trigram_model, "to", "be", top_k=3)
    )

    show_predictions(
        "TRIGRAM  |  context: 'and all'",
        predict_trigram(trigram_model, "and", "all", top_k=3)
    )

Tokens: 310 | Unique words: 172

────────────────────────────────────────
  UNIGRAM  →  most frequent words overall
────────────────────────────────────────
  1. 'the' █ 6.1%
  2. 'to' █ 5.5%
  3. 'and' █ 5.2%

────────────────────────────────────────
  BIGRAM   |  context: 'the'
────────────────────────────────────────
  1. 'question' █ 5.3%
  2. 'mind' █ 5.3%
  3. 'slings' █ 5.3%

────────────────────────────────────────
  BIGRAM   |  context: 'to'
────────────────────────────────────────
  1. 'be' █████ 17.6%
  2. 'die' ███ 11.8%
  3. 'sleep' ███ 11.8%

────────────────────────────────────────
  TRIGRAM  |  context: 'to be'
────────────────────────────────────────
  1. 'or' ██████████ 33.3%
  2. 'that' ██████████ 33.3%
  3. 'wished' ██████████ 33.3%

────────────────────────────────────────
  TRIGRAM  |  context: 'and all'
────────────────────────────────────────
  1. 'the' ███████████████ 50.0%
  2. 'our' ███████████████ 50.0%


# Exercise 2: Spam/Ham Classification

In [ ]:
import re
import math
from collections import defaultdict

#  STOP WORDS LIST

STOP_WORDS = {
    'i', 'me', 'my', 'we', 'our', 'you', 'your', 'he', 'him', 'his',
    'she', 'her', 'it', 'its', 'they', 'them', 'their', 'what', 'which',
    'who', 'this', 'that', 'these', 'those', 'am', 'is', 'are', 'was',
    'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does',
    'did', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'as', 'of',
    'at', 'by', 'for', 'with', 'to', 'from', 'in', 'out', 'on', 'up',
    'then', 'when', 'where', 'how', 'all', 'any', 'no', 'not', 'so',
    'than', 'too', 'can', 'will', 'just', 'should', 'now', 's', 't'
}

In [17]:
#STEP 1: Preprocess Text 
def preprocess_text(text, remove_stopwords=True):
    """Lowercase, remove punctuation, optionally remove stop words."""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)   # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    if remove_stopwords:
        words = [w for w in words if w not in STOP_WORDS]

    return words   # return list of words

In [19]:
#STEP 2: Load SMS Data
def load_sms_data(filepath):
    """Load SMSSpamCollection file → (messages, labels)"""
    messages, labels = [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 2:
                labels.append(parts[0])    # 'spam' or 'ham'
                messages.append(parts[1])  # message text
    return messages, labels

In [21]:
#STEP 3: Calculate Prior Probabilities
def calculate_priors(labels):
    """
    P(spam) = number of spam messages / total messages
    P(ham)  = number of ham  messages / total messages
    """
    total = len(labels)
    spam_count = labels.count('spam')
    ham_count  = labels.count('ham')

    prior_prob = {
        'spam': spam_count / total,
        'ham':  ham_count  / total
    }
    return prior_prob


In [22]:
#  STEP 4: Count Words Per Class 
def count_words(messages, labels):
    """
    Count how often each word appears in spam vs ham.
    Returns:
      word_counts  →  {'spam': {'free': 3, ...}, 'ham': {'ok': 5, ...}}
      vocab        →  set of all unique words
    """
    word_counts = {
        'spam': defaultdict(int),
        'ham':  defaultdict(int)
    }
    vocab = set()

    for message, label in zip(messages, labels):
        words = preprocess_text(message, remove_stopwords=True)
        for word in words:
            word_counts[label][word] += 1
            vocab.add(word)

    return word_counts, vocab

In [24]:
#  STEP 5: Compute Word Probabilities
#          (Laplace Smoothing) 
def calculate_word_probs(word_counts, vocab, k=1):
    """
    P(word | spam) = (count of word in spam + k)
                     ─────────────────────────────────────
                     (total words in spam + k * vocab_size)

    k=1 is called Laplace smoothing — prevents zero probabilities
    for words never seen in training.
    """
    V = len(vocab)   # vocabulary size

    word_probs = {'spam': {}, 'ham': {}}

    for label in ['spam', 'ham']:
        total_words = sum(word_counts[label].values())  # total word count for this class

        for word in vocab:
            count = word_counts[label][word]            # 0 if word never appeared
            word_probs[label][word] = (count + k) / (total_words + k * V)

    return word_probs

In [26]:
#  STEP 6: Predict Spam Probability 
def predict(message, prior_prob, word_probs, vocab):
    """
    Naive Bayes formula (in log space to avoid underflow):

    log P(spam | message) ∝ log P(spam) + Σ log P(word | spam)

    Steps:
      1. Start with log of prior probability
      2. For each word in message, add log(P(word|class))
      3. Convert back with exp()
      4. Normalize: P(spam) / (P(spam) + P(ham))
    """
    words = preprocess_text(message, remove_stopwords=True)

    # Start with log priors
    log_prob_spam = math.log(prior_prob['spam'])
    log_prob_ham  = math.log(prior_prob['ham'])

    # Add log likelihood of each word
    for word in words:
        if word in vocab:
            log_prob_spam += math.log(word_probs['spam'][word])
            log_prob_ham  += math.log(word_probs['ham'][word])

    # Convert back from log space
    prob_spam = math.exp(log_prob_spam)
    prob_ham  = math.exp(log_prob_ham)

    # Normalize to get probability between 0 and 1
    spam_probability = prob_spam / (prob_spam + prob_ham)
    return spam_probability


In [29]:
#  MAIN — Test Everything 
if __name__ == "__main__":

    # ── Small test data (no file needed) ──────────────────────────────────
    train_messages = [
        "Free prize winner click now",
        "Win free cash prize today",
        "Congratulations free offer claim now",
        "Hey are you coming to the meeting today",
        "Ok sounds good see you later",
        "Call me when you get home",
        "Free entry win a prize now",
        "Lunch meeting at noon today",
    ]
    train_labels = ['spam', 'spam', 'spam', 'ham', 'ham', 'ham', 'spam', 'ham']

    # ── Step 3: Priors ────────────────────────────────────────────────────
    prior_prob = calculate_priors(train_labels)
    print("=" * 45)
    print("STEP 3 — Prior Probabilities")
    print("=" * 45)
    print(f"  P(spam) = {prior_prob['spam']:.2f}")
    print(f"  P(ham)  = {prior_prob['ham']:.2f}")

    # ── Step 4: Word Counts ───────────────────────────────────────────────
    word_counts, vocab = count_words(train_messages, train_labels)
    print("\n" + "=" * 45)
    print("STEP 4 — Word Counts")
    print("=" * 45)
    print(f"  Vocabulary size : {len(vocab)} unique words")
    print(f"  'free' in spam  : {word_counts['spam']['free']}")
    print(f"  'free' in ham   : {word_counts['ham']['free']}")
    print(f"  'meeting' in ham: {word_counts['ham']['meeting']}")

    # ── Step 5: Word Probabilities ────────────────────────────────────────
    word_probs = calculate_word_probs(word_counts, vocab, k=1)
    print("\n" + "=" * 45)
    print("STEP 5 — Word Probabilities (with smoothing)")
    print("=" * 45)
    print(f"  P('free'  | spam) = {word_probs['spam']['free']:.4f}")
    print(f"  P('free'  | ham)  = {word_probs['ham']['free']:.4f}")
    print(f"  P('prize' | spam) = {word_probs['spam']['prize']:.4f}")
    print(f"  P('lunch' | ham)  = {word_probs['ham']['lunch']:.4f}")

    # ── Step 6: Predictions ───────────────────────────────────────────────
    test_messages = [
        "Free prize winner click now",   # obvious spam
        "See you at the meeting",        # obvious ham
        "Congratulations you won cash",  # spam
        "Are you free for lunch today",  # ham (tricky — contains 'free')
    ]

    print("\n" + "=" * 45)
    print("STEP 6 — Predictions")
    print("=" * 45)
    for msg in test_messages:
        prob = predict(msg, prior_prob, word_probs, vocab)
        label = "SPAM" if prob > 0.5 else "HAM "
        bar = "█" * int(prob * 20)
        print(f"  [{label}] {prob*100:5.1f}%  {bar:<20}  \"{msg}\"")

    # ── Stop Word Effect Demo ─────────────────────────────────────────────
    sample = "You have won a free prize today"
    with_sw    = preprocess_text(sample, remove_stopwords=False)
    without_sw = preprocess_text(sample, remove_stopwords=True)
    print("\n" + "=" * 45)
    print("BONUS — Stop Word Removal Effect")
    print("=" * 45)
    print(f"  Original      : {sample}")
    print(f"  With stops    : {with_sw}")
    print(f"  Without stops : {without_sw}")
    reduction = (1 - len(without_sw) / len(with_sw)) * 100
    print(f"  Reduction     : {reduction:.0f}% fewer words")
 

STEP 3 — Prior Probabilities
  P(spam) = 0.50
  P(ham)  = 0.50

STEP 4 — Word Counts
  Vocabulary size : 24 unique words
  'free' in spam  : 4
  'free' in ham   : 0
  'meeting' in ham: 2

STEP 5 — Word Probabilities (with smoothing)
  P('free'  | spam) = 0.1220
  P('free'  | ham)  = 0.0250
  P('prize' | spam) = 0.0976
  P('lunch' | ham)  = 0.0500

STEP 6 — Predictions
  [SPAM]  98.6%  ███████████████████   "Free prize winner click now"
  [HAM ]  13.7%  ██                    "See you at the meeting"
  [SPAM]  79.2%  ███████████████       "Congratulations you won cash"
  [SPAM]  60.7%  ████████████          "Are you free for lunch today"

BONUS — Stop Word Removal Effect
  Original      : You have won a free prize today
  With stops    : ['you', 'have', 'won', 'a', 'free', 'prize', 'today']
  Without stops : ['won', 'free', 'prize', 'today']
  Reduction     : 43% fewer words
